# GIFT × Riemann : Phase 3 - Calcul Direct des Zéros

**Ce notebook calcule les zéros des L-functions au lieu de les télécharger.**

Fonctionnalités :
- Calcul des zéros de L-functions de Dirichlet (mpmath)
- Test massif GIFT vs non-GIFT conductors
- Alternative Ramanujan via approximation

---

In [ ]:
# ============================================================
# CELLULE 1 : INSTALLATION ET SETUP
# ============================================================

!pip install -q mpmath sympy

import numpy as np
from mpmath import mp, mpf, mpc, gamma as mpgamma, zeta, pi as mppi, exp, log, sin, cos, sqrt
from mpmath import fsum, nsum, inf
import json
import os
from typing import List, Tuple, Dict, Callable
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Précision mpmath
mp.dps = 25  # 25 décimales

# Constantes GIFT
@dataclass
class GIFT:
    b3: int = 77
    dim_K7: int = 7
    dim_G2: int = 14
    h_G2: int = 6
    rank_E8: int = 8
    dim_E8: int = 248
    b2: int = 21
    dim_J3O: int = 27
    H_star: int = 99
    h_G2_sq: int = 36
    gift_lags: tuple = (5, 8, 13, 27)
    std_lags: tuple = (1, 2, 3, 4)
    
    # Conducteurs à tester
    gift_conductors: tuple = (5, 7, 8, 13, 14, 21, 27, 77, 99)
    nongift_conductors: tuple = (10, 11, 15, 16, 19, 23, 29, 31, 37)
    border_conductors: tuple = (6, 9, 12, 20, 22, 26, 28, 76, 78)

G = GIFT()
DATA = {}
RESULTS = {'meta': {'phase': 3, 'method': 'compute'}, 'tests': {}}

print("✓ Setup complet")
print(f"  Précision: {mp.dps} décimales")
print(f"  GIFT: {G.gift_conductors}")
print(f"  Non-GIFT: {G.nongift_conductors}")
print(f"  Border: {G.border_conductors}")

In [ ]:
# ============================================================
# CELLULE 2 : CARACTÈRES DE DIRICHLET
# ============================================================

def gcd(a: int, b: int) -> int:
    """Plus grand commun diviseur."""
    while b:
        a, b = b, a % b
    return a

def euler_phi(n: int) -> int:
    """Fonction d'Euler φ(n)."""
    result = 0
    for k in range(1, n + 1):
        if gcd(k, n) == 1:
            result += 1
    return result

def primitive_root(p: int) -> int:
    """Trouve une racine primitive modulo p (p premier)."""
    if p == 2:
        return 1
    phi = p - 1
    factors = []
    n = phi
    d = 2
    while d * d <= n:
        if n % d == 0:
            factors.append(d)
            while n % d == 0:
                n //= d
        d += 1
    if n > 1:
        factors.append(n)
    
    for g in range(2, p):
        is_primitive = True
        for f in factors:
            if pow(g, phi // f, p) == 1:
                is_primitive = False
                break
        if is_primitive:
            return g
    return 2

def dirichlet_character(q: int, a: int) -> Callable[[int], complex]:
    """
    Construit le a-ème caractère de Dirichlet mod q.
    Pour q premier, utilise les puissances de racines primitives.
    """
    if q <= 1:
        return lambda n: 1
    
    phi_q = euler_phi(q)
    zeta_phi = mpc(cos(2 * mppi / phi_q), sin(2 * mppi / phi_q))
    
    # Table de logarithmes discrets (simple pour petits q)
    if q <= 500:
        # Pour q premier
        if all(q % p != 0 for p in range(2, int(q**0.5) + 1)):
            g = primitive_root(q)
            log_table = {pow(g, k, q): k for k in range(phi_q)}
            
            def chi(n: int) -> complex:
                n = n % q
                if n == 0 or gcd(n, q) != 1:
                    return mpc(0)
                k = log_table.get(n, 0)
                return zeta_phi ** (a * k)
            return chi
    
    # Fallback: caractère principal
    def chi_principal(n: int) -> complex:
        return mpc(1) if gcd(n, q) == 1 else mpc(0)
    return chi_principal

print("✓ Caractères de Dirichlet implémentés")

# Test
chi = dirichlet_character(5, 1)
print(f"  χ₅(1) = {chi(1)}")
print(f"  χ₅(2) = {chi(2)}")
print(f"  χ₅(5) = {chi(5)}")

In [ ]:
# ============================================================
# CELLULE 3 : FONCTION L DE DIRICHLET
# ============================================================

def dirichlet_L(s: complex, chi: Callable, q: int, terms: int = 1000) -> complex:
    """
    Calcule L(s, χ) = Σ χ(n)/n^s pour Re(s) > 1.
    Pour Re(s) ≤ 1, utilise le prolongement via équation fonctionnelle.
    """
    s = mpc(s)
    
    if s.real > 1:
        # Série directe convergente
        total = mpc(0)
        for n in range(1, terms + 1):
            chi_n = chi(n)
            if chi_n != 0:
                total += chi_n / (mpf(n) ** s)
        return total
    else:
        # Utiliser la formule d'Hurwitz pour continuation
        # L(s, χ) = q^(-s) Σ_{a=1}^{q} χ(a) ζ(s, a/q)
        total = mpc(0)
        for a in range(1, q + 1):
            chi_a = chi(a)
            if chi_a != 0:
                # ζ(s, a/q) via série
                hurwitz_sum = mpc(0)
                for n in range(terms):
                    hurwitz_sum += mpf(1) / ((n + mpf(a)/q) ** s)
                total += chi_a * hurwitz_sum
        return total * (mpf(q) ** (-s))

def L_on_critical_line(t: float, chi: Callable, q: int) -> complex:
    """
    Calcule L(1/2 + it, χ).
    """
    s = mpc(mpf('0.5'), mpf(t))
    return dirichlet_L(s, chi, q, terms=500)

print("✓ Fonction L de Dirichlet implémentée")

# Test avec caractère trivial (devrait donner ζ(s))
chi_trivial = lambda n: mpc(1)
print(f"  L(2, χ₀) ≈ {dirichlet_L(2, chi_trivial, 1, 1000):.6f}")
print(f"  ζ(2) = π²/6 ≈ {float(mppi**2/6):.6f}")

In [ ]:
# ============================================================
# CELLULE 4 : RECHERCHE DES ZÉROS
# ============================================================

def find_zeros_simple(L_func: Callable, t_min: float, t_max: float, 
                      step: float = 0.5, refine: bool = True) -> List[float]:
    """
    Trouve les zéros par changement de signe de la partie réelle.
    Méthode simple mais robuste.
    """
    zeros = []
    t = t_min
    prev_val = L_func(t)
    prev_sign = 1 if prev_val.real > 0 else -1
    
    while t < t_max:
        t += step
        try:
            val = L_func(t)
            sign = 1 if val.real > 0 else -1
            
            # Changement de signe → zéro entre t-step et t
            if sign != prev_sign:
                if refine:
                    # Bisection pour affiner
                    a, b = t - step, t
                    for _ in range(20):  # ~6 décimales
                        mid = (a + b) / 2
                        val_mid = L_func(mid)
                        sign_mid = 1 if val_mid.real > 0 else -1
                        if sign_mid == prev_sign:
                            a = mid
                        else:
                            b = mid
                    zero = (a + b) / 2
                else:
                    zero = t - step/2
                
                zeros.append(float(zero))
            
            prev_sign = sign
            prev_val = val
        except:
            pass
    
    return zeros

def compute_zeros_for_conductor(q: int, n_zeros: int = 100, 
                                 char_index: int = 1,
                                 verbose: bool = True) -> np.ndarray:
    """
    Calcule les premiers zéros de L(s, χ) pour le conducteur q.
    """
    if verbose:
        print(f"  Calcul q={q}...", end=' ', flush=True)
    
    chi = dirichlet_character(q, char_index)
    L_func = lambda t: L_on_critical_line(t, chi, q)
    
    # Estimer t_max nécessaire (heuristique)
    # Densité des zéros ≈ (1/2π) log(q*t/2π)
    t_max = max(100, n_zeros * 3)  # Marge de sécurité
    
    zeros = find_zeros_simple(L_func, 1.0, t_max, step=0.3)
    
    if verbose:
        print(f"{len(zeros)} zéros trouvés")
    
    return np.array(zeros[:n_zeros]) if len(zeros) >= n_zeros else np.array(zeros)

print("✓ Algorithme de recherche des zéros prêt")

In [ ]:
# ============================================================
# CELLULE 5 : TEST RAPIDE - UN CONDUCTEUR
# ============================================================

print("="*60)
print("TEST RAPIDE : Calcul zéros pour q=5")
print("="*60)

# Test avec q=5 (premier conducteur GIFT)
zeros_q5 = compute_zeros_for_conductor(5, n_zeros=50, verbose=True)

if len(zeros_q5) >= 10:
    print(f"\nPremiers zéros: {zeros_q5[:10].round(2)}")
    print(f"Espacement moyen: {np.mean(np.diff(zeros_q5)):.2f}")
    DATA['L_q5'] = zeros_q5
else:
    print("⚠️ Pas assez de zéros trouvés")

In [ ]:
# ============================================================
# CELLULE 6 : CALCUL BATCH - TOUS LES CONDUCTEURS
# ============================================================

print("="*60)
print("CALCUL BATCH : GIFT vs NON-GIFT vs BORDER")
print("="*60)

# Paramètres
N_ZEROS_TARGET = 80  # Zéros par conducteur

# Tous les conducteurs à tester
all_conductors = (
    list(G.gift_conductors) + 
    list(G.nongift_conductors) + 
    list(G.border_conductors)
)
all_conductors = sorted(set(all_conductors))

print(f"\nConducteurs à tester: {all_conductors}")
print(f"Zéros cibles par conducteur: {N_ZEROS_TARGET}")
print()

for q in all_conductors:
    key = f'L_q{q}'
    if key not in DATA:
        try:
            zeros = compute_zeros_for_conductor(q, n_zeros=N_ZEROS_TARGET, verbose=True)
            if len(zeros) >= 30:  # Minimum pour analyse
                DATA[key] = zeros
        except Exception as e:
            print(f"  ✗ q={q}: {e}")

print(f"\n✓ {len([k for k in DATA if k.startswith('L_q')])} conducteurs calculés")

In [ ]:
# ============================================================
# CELLULE 7 : CHARGEMENT ZÉROS RIEMANN (Odlyzko)
# ============================================================

print("="*60)
print("ZÉROS DE RIEMANN (référence)")
print("="*60)

# Essayer de télécharger les zéros d'Odlyzko
import urllib.request

ODLYZKO_URL = 'https://www.dtc.umn.edu/~odlyzko/zeta_tables/zeros1'
CACHE_FILE = 'odlyzko_zeros.txt'

def load_riemann_zeros():
    """Charge les zéros de Riemann depuis Odlyzko ou calcule les premiers."""
    
    # Essayer le cache
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r') as f:
            zeros = [float(line.strip()) for line in f if line.strip()]
        if len(zeros) > 100:
            return np.array(zeros)
    
    # Essayer le téléchargement
    try:
        print("  Téléchargement Odlyzko...", end=' ')
        req = urllib.request.Request(ODLYZKO_URL, headers={'User-Agent': 'GIFT Research'})
        with urllib.request.urlopen(req, timeout=30) as response:
            content = response.read().decode('utf-8')
        
        zeros = []
        for line in content.split('\n'):
            line = line.strip()
            if line and not line.startswith('#'):
                try:
                    zeros.append(float(line.split()[0]))
                except:
                    pass
        
        if len(zeros) > 100:
            with open(CACHE_FILE, 'w') as f:
                f.write('\n'.join(str(z) for z in zeros))
            print(f"{len(zeros)} zéros")
            return np.array(zeros)
    except Exception as e:
        print(f"Échec: {e}")
    
    # Fallback: calculer les premiers zéros avec mpmath
    print("  Calcul avec mpmath...")
    from mpmath import zetazero
    zeros = [float(zetazero(n).imag) for n in range(1, 201)]
    return np.array(zeros)

zeros_riemann = load_riemann_zeros()
DATA['zeta'] = zeros_riemann
print(f"\n  ζ(s): {len(zeros_riemann)} zéros chargés")
print(f"  γ₁ = {zeros_riemann[0]:.4f}")
print(f"  γ₁₀ = {zeros_riemann[9]:.4f}")

In [ ]:
# ============================================================
# CELLULE 8 : ANALYSE - FIT RÉCURRENCE
# ============================================================

print("="*60)
print("ANALYSE : FIT RÉCURRENCE GIFT vs STANDARD")
print("="*60)

def fit_recurrence(gamma: np.ndarray, lags: List[int]) -> Tuple[np.ndarray, float, float]:
    """Fit récurrence linéaire et retourne coeffs, erreur, R²."""
    max_lag = max(lags)
    start = max_lag + 5
    end = len(gamma)
    
    if end - start < 30:
        return None, float('inf'), 0
    
    n_points = end - start
    X = np.zeros((n_points, len(lags) + 1))
    for i, lag in enumerate(lags):
        X[:, i] = gamma[start - lag:end - lag]
    X[:, -1] = 1.0  # Intercept
    
    y = gamma[start:end]
    
    # Moindres carrés
    coeffs, residuals, _, _ = np.linalg.lstsq(X, y, rcond=None)
    
    y_pred = X @ coeffs
    mse = float(np.mean((y_pred - y)**2))
    mae = float(np.mean(np.abs(y_pred - y)))
    
    # R²
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    
    return coeffs, mae, float(r2)

def analyze_zeros(zeros: np.ndarray, name: str) -> Dict:
    """Analyse complète d'un jeu de zéros."""
    result = {'name': name, 'n_zeros': len(zeros)}
    
    if len(zeros) < 40:
        result['status'] = 'insufficient'
        return result
    
    # GIFT lags
    coeffs_gift, err_gift, r2_gift = fit_recurrence(zeros, list(G.gift_lags))
    coeffs_std, err_std, r2_std = fit_recurrence(zeros, list(G.std_lags))
    
    if coeffs_gift is None:
        result['status'] = 'fit_failed'
        return result
    
    # Produits lag × coeff
    products = {lag: lag * coeffs_gift[i] for i, lag in enumerate(G.gift_lags)}
    
    # Ratio Fibonacci: (8×a₈)/(13×a₁₃)
    ratio = products[8] / products[13] if products[13] != 0 else float('inf')
    
    result.update({
        'status': 'success',
        'coeffs_gift': [float(c) for c in coeffs_gift],
        'coeffs_std': [float(c) for c in coeffs_std] if coeffs_std is not None else None,
        'gift_error': err_gift,
        'std_error': err_std,
        'gift_r2': r2_gift,
        'std_r2': r2_std,
        'gift_wins': err_gift < err_std,
        'improvement': (err_std - err_gift) / err_std * 100 if err_std > 0 else 0,
        'products': {str(k): float(v) for k, v in products.items()},
        'ratio_8_13': float(ratio),
        'deviation': float(abs(ratio - 1))
    })
    
    return result

print("✓ Fonctions d'analyse prêtes")

In [ ]:
# ============================================================
# CELLULE 9 : ANALYSE DE TOUS LES DATASETS
# ============================================================

print("="*60)
print("ANALYSE COMPLÈTE")
print("="*60)

all_results = []

for name, zeros in sorted(DATA.items()):
    result = analyze_zeros(zeros, name)
    
    # Classifier GIFT/non-GIFT
    if name.startswith('L_q'):
        q = int(name.replace('L_q', ''))
        result['conductor'] = q
        result['category'] = 'GIFT' if q in G.gift_conductors else (
            'NON-GIFT' if q in G.nongift_conductors else 'BORDER'
        )
    elif name == 'zeta':
        result['category'] = 'REFERENCE'
    else:
        result['category'] = 'OTHER'
    
    all_results.append(result)
    
    if result['status'] == 'success':
        cat = result['category']
        dev = result['deviation'] * 100
        wins = '✓' if result['gift_wins'] else '✗'
        print(f"  {name:<12} [{cat:<8}] |R-1| = {dev:6.1f}%  GIFT wins: {wins}")

RESULTS['all_results'] = all_results

In [ ]:
# ============================================================
# CELLULE 10 : STATISTIQUES GIFT vs NON-GIFT vs BORDER
# ============================================================

print("="*60)
print("STATISTIQUES PAR CATÉGORIE")
print("="*60)

# Séparer par catégorie
categories = {}
for r in all_results:
    if r['status'] == 'success':
        cat = r['category']
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(r)

# Statistiques
stats = {}
for cat, results in categories.items():
    devs = [r['deviation'] for r in results]
    wins = sum(1 for r in results if r['gift_wins'])
    
    stats[cat] = {
        'n': len(results),
        'mean_dev': np.mean(devs),
        'std_dev': np.std(devs),
        'min_dev': np.min(devs),
        'max_dev': np.max(devs),
        'gift_wins': wins,
        'win_rate': wins / len(results) * 100
    }
    
    print(f"\n{cat} ({len(results)} conducteurs):")
    print(f"  Déviation moyenne: {np.mean(devs)*100:.1f}%")
    print(f"  Écart-type: {np.std(devs)*100:.1f}%")
    print(f"  Min-Max: {np.min(devs)*100:.1f}% - {np.max(devs)*100:.1f}%")
    print(f"  GIFT gagne: {wins}/{len(results)} ({wins/len(results)*100:.0f}%)")

RESULTS['stats'] = stats

In [ ]:
# ============================================================
# CELLULE 11 : TEST DE SÉLECTIVITÉ (t-test)
# ============================================================

print("="*60)
print("TEST STATISTIQUE : SÉLECTIVITÉ GIFT")
print("="*60)

from scipy import stats as scipy_stats

# Extraire les déviations
gift_devs = [r['deviation'] for r in all_results 
             if r['status'] == 'success' and r['category'] == 'GIFT']
nongift_devs = [r['deviation'] for r in all_results 
                if r['status'] == 'success' and r['category'] == 'NON-GIFT']

print(f"\nÉchantillons:")
print(f"  GIFT: n={len(gift_devs)}, μ={np.mean(gift_devs)*100:.1f}%")
print(f"  NON-GIFT: n={len(nongift_devs)}, μ={np.mean(nongift_devs)*100:.1f}%")

if len(gift_devs) >= 3 and len(nongift_devs) >= 3:
    # t-test (one-tailed: GIFT < NON-GIFT)
    t_stat, p_value = scipy_stats.ttest_ind(gift_devs, nongift_devs, alternative='less')
    
    # Mann-Whitney U (non-paramétrique)
    u_stat, p_value_mw = scipy_stats.mannwhitneyu(gift_devs, nongift_devs, alternative='less')
    
    # Ratio de sélectivité
    selectivity = np.mean(nongift_devs) / np.mean(gift_devs) if np.mean(gift_devs) > 0 else float('inf')
    
    print(f"\nTests statistiques:")
    print(f"  t-test: t={t_stat:.2f}, p={p_value:.4f}")
    print(f"  Mann-Whitney: U={u_stat:.1f}, p={p_value_mw:.4f}")
    print(f"\nRatio de sélectivité: {selectivity:.1f}×")
    
    # Verdict
    if p_value < 0.05:
        print(f"\n🎉 SÉLECTIVITÉ CONFIRMÉE (p < 0.05)")
        print(f"   Les conducteurs GIFT ont une déviation significativement plus faible.")
    elif p_value < 0.10:
        print(f"\n⚠️ TENDANCE (p < 0.10)")
        print(f"   Signal suggestif mais pas statistiquement significatif.")
    else:
        print(f"\n❌ PAS DE DIFFÉRENCE SIGNIFICATIVE (p = {p_value:.2f})")
    
    RESULTS['selectivity_test'] = {
        'n_gift': len(gift_devs),
        'n_nongift': len(nongift_devs),
        'mean_gift': float(np.mean(gift_devs)),
        'mean_nongift': float(np.mean(nongift_devs)),
        't_stat': float(t_stat),
        'p_value': float(p_value),
        'selectivity_ratio': float(selectivity),
        'significant': p_value < 0.05
    }
else:
    print("\n⚠️ Pas assez de données pour le test statistique")

In [ ]:
# ============================================================
# CELLULE 12 : TABLEAU RÉCAPITULATIF
# ============================================================

print("="*70)
print("CLASSEMENT COMPLET PAR DÉVIATION FIBONACCI")
print("="*70)

# Trier par déviation
sorted_results = sorted(
    [r for r in all_results if r['status'] == 'success'],
    key=lambda x: x['deviation']
)

print(f"\n{'Rang':<5} {'Dataset':<12} {'Cat':<10} {'N':<6} {'|R-1|':<10} {'GIFT?':<6}")
print("-" * 55)

for i, r in enumerate(sorted_results, 1):
    name = r['name']
    cat = r['category']
    n = r['n_zeros']
    dev = f"{r['deviation']*100:.1f}%"
    wins = '✓' if r['gift_wins'] else ''
    
    # Marqueur visuel
    if cat == 'GIFT':
        marker = '★'
    elif cat == 'NON-GIFT':
        marker = '✗'
    else:
        marker = '○'
    
    print(f"{i:<5} {name:<12} {cat:<10} {n:<6} {dev:<10} {marker}")

In [ ]:
# ============================================================
# CELLULE 13 : VISUALISATION
# ============================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Box plot par catégorie
ax1 = axes[0]
cat_data = []
cat_labels = []
colors = []

for cat in ['GIFT', 'BORDER', 'NON-GIFT']:
    if cat in categories:
        devs = [r['deviation'] * 100 for r in categories[cat]]
        cat_data.append(devs)
        cat_labels.append(f"{cat}\n(n={len(devs)})")
        colors.append('green' if cat == 'GIFT' else 'orange' if cat == 'BORDER' else 'red')

bp = ax1.boxplot(cat_data, labels=cat_labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)

ax1.set_ylabel('Déviation |R-1| (%)')
ax1.set_title('Déviation Fibonacci par Catégorie')
ax1.axhline(y=0, color='black', linestyle='--', alpha=0.3)

# 2. Scatter plot: déviation vs conducteur
ax2 = axes[1]

for r in sorted_results:
    if 'conductor' in r:
        q = r['conductor']
        dev = r['deviation'] * 100
        cat = r['category']
        
        color = 'green' if cat == 'GIFT' else 'orange' if cat == 'BORDER' else 'red'
        marker = 's' if cat == 'GIFT' else 'o' if cat == 'BORDER' else 'x'
        
        ax2.scatter(q, dev, c=color, marker=marker, s=80, alpha=0.7)
        ax2.annotate(str(q), (q, dev), textcoords="offset points", 
                    xytext=(0,5), ha='center', fontsize=8)

ax2.set_xlabel('Conducteur q')
ax2.set_ylabel('Déviation |R-1| (%)')
ax2.set_title('Déviation vs Conducteur')
ax2.legend(['GIFT ★', 'BORDER ○', 'NON-GIFT ✗'], loc='upper right')

plt.tight_layout()
plt.savefig('phase3_selectivity.png', dpi=150)
plt.show()

print("\n✓ Figure sauvegardée: phase3_selectivity.png")

In [ ]:
# ============================================================
# CELLULE 14 : SYNTHÈSE FINALE
# ============================================================

print("="*70)
print("🎯 SYNTHÈSE PHASE 3 - TEST MASSIF SÉLECTIVITÉ")
print("="*70)

print(f"\n📊 DONNÉES")
print(f"   {len(DATA)} datasets analysés")
print(f"   GIFT: {len(gift_devs)} conducteurs")
print(f"   NON-GIFT: {len(nongift_devs)} conducteurs")

print(f"\n📈 RÉSULTATS")
if 'selectivity_test' in RESULTS:
    st = RESULTS['selectivity_test']
    print(f"   Déviation GIFT: {st['mean_gift']*100:.1f}%")
    print(f"   Déviation non-GIFT: {st['mean_nongift']*100:.1f}%")
    print(f"   Ratio sélectivité: {st['selectivity_ratio']:.1f}×")
    print(f"   p-value: {st['p_value']:.4f}")
    
    if st['significant']:
        print(f"\n   ✅ SÉLECTIVITÉ STATISTIQUEMENT SIGNIFICATIVE")
    else:
        print(f"\n   ⚠️ Tendance observée mais non significative")

print(f"\n📋 PROCHAINES ÉTAPES")
print("   1. Augmenter le nombre de zéros par conducteur (>500)")
print("   2. Ajouter plus de conducteurs non-GIFT")
print("   3. Calculer les zéros Ramanujan (SageMath)")
print("   4. Analyse RG flow sur datasets étendus")

# Sauvegarder
RESULTS['meta']['n_datasets'] = len(DATA)
RESULTS['meta']['status'] = 'complete'

with open('phase3_compute_results.json', 'w') as f:
    json.dump(RESULTS, f, indent=2, default=str)

print(f"\n✓ Résultats sauvegardés: phase3_compute_results.json")

---

## Note sur Ramanujan Delta

Pour calculer les zéros de la L-function de Ramanujan (>1000 zéros), il faut:

1. **SageMath** (recommandé):
```python
from sage.modular.modform.constructor import ModularForms
M = ModularForms(Gamma0(1), 12)
delta = M.newforms()[0]
L = delta.lseries()
zeros = L.find_zeros(0, 10000)
```

2. **CoCalc** (gratuit): https://cocalc.com - SageMath en ligne

3. **Alternative mpmath** (lent mais possible):
   - Implémenter L(s, Δ) = Σ τ(n)/n^s avec les coefficients τ(n) de Ramanujan
   - Chercher les zéros sur Re(s) = 6

---